### 《实用Python编程》教学代码
## 第3章 数据存储与网上数据抓取-part2

## 示例代码3.20 使用Base64编码将URL转换为文件名

In [1]:
import base64

"""将 URL 编码为安全的文件名"""
def url_to_filename(url):
    url_bytes = url.encode('utf-8')
    base64_bytes = base64.urlsafe_b64encode(url_bytes)
    filename = base64_bytes.decode('utf-8')
    filename = filename.rstrip('=') # 移除可能存在的填充字符（=）
    return filename

"""将文件名解码回原始 URL"""
def filename_to_url(filename):
    padding = 4 - (len(filename) % 4) # 填充字符（=）回填
    if padding != 4:
        filename += '=' * padding
    base64_bytes = filename.encode('utf-8')
    url_bytes = base64.urlsafe_b64decode(base64_bytes)
    return url_bytes.decode('utf-8')

"""测试样例"""
test_urls = """https://gaokao.eol.cn/bei_jing/dongtai/202407/t20240720_2625168.shtml
https://gaokao.eol.cn/bei_jing/dongtai/202406/t20240625_2619192.shtml
https://www.dxsbb.com/news/46721.html
http://www.moe.gov.cn/srcsite/A22/s7065/200612/t20061206_128833.html
http://www.moe.gov.cn/srcsite/A22/s7065/200512/t20051223_82762.html
https://rdzs.ruc.edu.cn/inquiry/admission/indexcms
https://rdzs.ruc.edu.cn/inquiry/enrollplan/indexcms
https://m.tujia.com/hotel_city73""".split()
for ori_url in test_urls:
    filename = url_to_filename(ori_url)
    new_url = filename_to_url(filename)
    print(f"{ori_url} -> {filename}")
    assert ori_url == new_url

https://gaokao.eol.cn/bei_jing/dongtai/202407/t20240720_2625168.shtml -> aHR0cHM6Ly9nYW9rYW8uZW9sLmNuL2JlaV9qaW5nL2Rvbmd0YWkvMjAyNDA3L3QyMDI0MDcyMF8yNjI1MTY4LnNodG1s
https://gaokao.eol.cn/bei_jing/dongtai/202406/t20240625_2619192.shtml -> aHR0cHM6Ly9nYW9rYW8uZW9sLmNuL2JlaV9qaW5nL2Rvbmd0YWkvMjAyNDA2L3QyMDI0MDYyNV8yNjE5MTkyLnNodG1s
https://www.dxsbb.com/news/46721.html -> aHR0cHM6Ly93d3cuZHhzYmIuY29tL25ld3MvNDY3MjEuaHRtbA
http://www.moe.gov.cn/srcsite/A22/s7065/200612/t20061206_128833.html -> aHR0cDovL3d3dy5tb2UuZ292LmNuL3NyY3NpdGUvQTIyL3M3MDY1LzIwMDYxMi90MjAwNjEyMDZfMTI4ODMzLmh0bWw
http://www.moe.gov.cn/srcsite/A22/s7065/200512/t20051223_82762.html -> aHR0cDovL3d3dy5tb2UuZ292LmNuL3NyY3NpdGUvQTIyL3M3MDY1LzIwMDUxMi90MjAwNTEyMjNfODI3NjIuaHRtbA
https://rdzs.ruc.edu.cn/inquiry/admission/indexcms -> aHR0cHM6Ly9yZHpzLnJ1Yy5lZHUuY24vaW5xdWlyeS9hZG1pc3Npb24vaW5kZXhjbXM
https://rdzs.ruc.edu.cn/inquiry/enrollplan/indexcms -> aHR0cHM6Ly9yZHpzLnJ1Yy5lZHUuY24vaW5xdWlyeS9lbnJvbGxwbGFuL2luZGV4Y21z
http

## 课堂练习-3-2-1
将复旦大学历年录取分数网站的URL转换为文件名，再将文件名还原为URL 
+ https://ao.fudan.edu.cn/36333/list.htm

## 示例代码3.21 使用requests发送HTTP请求

In [2]:
import requests

target_url = "http://www.moe.gov.cn/srcsite/A22/s7065/200612/t20061206_128833.html"
response = requests.get(target_url, headers={"User-Agent": "Mozilla/5.0"})
print(response.status_code, response.encoding) # 输出 200 ISO-8859-1

200 ISO-8859-1


## 示例代码3.22 尝试向不存在的 URL 发送请求

In [3]:
import requests

try:
    response = requests.get("http://www.doesnotexist.html")
    print(response.status_code)
except Exception as e:
    print(e)

HTTPConnectionPool(host='www.doesnotexist.html', port=80): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPConnection object at 0x120c71310>: Failed to resolve 'www.doesnotexist.html' ([Errno 8] nodename nor servname provided, or not known)"))


## 示例代码3.23 定义一个页面下载类 PageDownloader 取单个静态网页

In [5]:
import os

class PageDownloader:
    def __init__(self, save_dir="html_pages"):
        self.save_dir = save_dir
        os.makedirs(self.save_dir, exist_ok=True) 
    def crawl(self, url):
        try:
            response = requests.get(url,  headers={"User-Agent": "Mozilla/5.0"})
            response.raise_for_status()  # 状态码异常处理
            return response
        except requests.exceptions.RequestException as e:
            print(f"[Error] 获取页面失败: {e}")
            return None
    def save(self, response, filename):
        try:
            if response.encoding.lower() == 'none': # 检测并设置正确的文本编码
                response.encoding = response.apparent_encoding
            with open(filename, 'w', encoding=response.encoding) as fw: # 保存内容
                fw.write(response.text)
            print(f"[Status] 已保存到 {filename}")
            return True
        except Exception as e:
            print(f"[Error] 保存页面失败: {e}")
            return False      
    def process(self, target_url):
        print(f"[Status] 开始抓取 {target_url}")
        response = self.crawl(target_url)
        if response and response.status_code == 200:
            filename = os.path.join(self.save_dir, url_to_filename(target_url) + '.html')
            self.save(response, filename)
        else:
            response = None
        message = "成功" if response else "失败"
        print(f"[Status] 页面抓取{message}")
        return response
"""测试代码"""
test_url = "http://www.moe.gov.cn/srcsite/A22/s7065/200612/t20061206_128833.html"
page = PageDownloader()
page.process(test_url)

[Status] 开始抓取 http://www.moe.gov.cn/srcsite/A22/s7065/200612/t20061206_128833.html
[Status] 已保存到 html_pages/aHR0cDovL3d3dy5tb2UuZ292LmNuL3NyY3NpdGUvQTIyL3M3MDY1LzIwMDYxMi90MjAwNjEyMDZfMTI4ODMzLmh0bWw.html
[Status] 页面抓取成功


<Response [200]>

## 课堂练习-3-2-2
用 PageDownloader 抓取复旦大学历年录取分数网页
+ https://ao.fudan.edu.cn/36333/list.htm


## 示例代码3.24 网络爬虫的基本框架

In [8]:
import time, random, requests
from urllib.parse import urlparse, urljoin
from bs4 import BeautifulSoup

class WebCrawler(PageDownloader):
    def __init__(self, trace_link=True, save_dir="html_pages"):
        super().__init__(save_dir)
        self.trace_link = trace_link
        self.task_queue = []
        self.url_cache = set() # 已访问 URL 集合，避免重复爬取

    def add(self, url):
        if url not in self.url_cache:
            self.task_queue.append(url)

    def addmany(self, urls):
        for url in urls:
            self.add(url)

    def fetch(self):
        print(f"[Status] 当前工作队列长度：{len(self.task_queue)}")
        if self.task_queue:
            return self.task_queue.pop(0)
        else:
            return None

    def parse(self, htmlstr, base_url):
        soup = BeautifulSoup(htmlstr, 'html.parser')
        new_links = set()
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href'].strip()
            if not href or href.startswith(('javascript:', 'mailto:', 'tel:')): # 跳过无效链接
                continue
            link = urljoin(base_url, href) # 规范化 URL（转换为绝对路径并移除片段标识符）
            parsed = urlparse(link)
            if parsed.fragment:
                link = parsed._replace(fragment='').geturl()
            new_links.add(link)
        return list(new_links)

    def process(self, url):
        response = super().process(url)
        self.url_cache.add(url)
        if response and self.trace_link:
            new_seeds = self.parse(response.text, url)
            self.addmany(new_seeds)
        return response

    def run(self):
        url = self.fetch()
        while url:
            res = self.process(url)
            if len(self.url_cache) >= 5: # 抓取了 5 个网页后就强制结束
                break
            sleep_time = random.uniform(1, 5)
            print(f"[Status] Sleeping for {sleep_time:.2f} seconds...")
            time.sleep(sleep_time) # 随机休眠，以免影响网站正常运行
            url = self.fetch()
        print(f"[Status] {self.__class__.__name__} stopped after {len(self.url_cache)} pages visited.")

"""测试代码"""
test_url = "http://www.moe.gov.cn/srcsite/A22/s7065/200612/t20061206_128833.html"
wc = WebCrawler()
wc.add(test_url)
wc.run()

[Status] 当前工作队列长度：1
[Status] 开始抓取 http://www.moe.gov.cn/srcsite/A22/s7065/200612/t20061206_128833.html
[Status] 已保存到 html_pages/aHR0cDovL3d3dy5tb2UuZ292LmNuL3NyY3NpdGUvQTIyL3M3MDY1LzIwMDYxMi90MjAwNjEyMDZfMTI4ODMzLmh0bWw.html
[Status] 页面抓取成功
[Status] Sleeping for 4.36 seconds...
[Status] 当前工作队列长度：28
[Status] 开始抓取 http://www.moe.gov.cn/jyb_sjzl/
[Status] 已保存到 html_pages/aHR0cDovL3d3dy5tb2UuZ292LmNuL2p5Yl9zanpsLw.html
[Status] 页面抓取成功
[Status] Sleeping for 4.46 seconds...
[Status] 当前工作队列长度：98
[Status] 开始抓取 https://rz.moe.gov.cn/tacs-uc/corp/register
[Status] 已保存到 html_pages/aHR0cHM6Ly9yei5tb2UuZ292LmNuL3RhY3MtdWMvY29ycC9yZWdpc3Rlcg.html
[Status] 页面抓取成功
[Status] Sleeping for 3.86 seconds...
[Status] 当前工作队列长度：114
[Status] 开始抓取 http://www.moe.gov.cn/jyb_xxgk/
[Status] 已保存到 html_pages/aHR0cDovL3d3dy5tb2UuZ292LmNuL2p5Yl94eGdrLw.html
[Status] 页面抓取成功
[Status] Sleeping for 2.96 seconds...
[Status] 当前工作队列长度：161
[Status] 开始抓取 http://ru.moe.gov.cn/
[Status] 已保存到 html_pages/aHR0cDovL3J1Lm1vZS5nb3YuY24

## 示例代码 3.25 使用网络爬虫批量抓取静态网页

### Step 1. 抓取根页面

In [9]:
base_url = "https://join-tsinghua.edu.cn/xxgk/lnlqfsx.htm"
wc = WebCrawler(trace_link=False, save_dir='thu')
response = wc.process(base_url)
response.encoding = response.apparent_encoding # 使用自动检测的编码，以免文本出现乱码

[Status] 开始抓取 https://join-tsinghua.edu.cn/xxgk/lnlqfsx.htm
[Status] 已保存到 thu/aHR0cHM6Ly9qb2luLXRzaW5naHVhLmVkdS5jbi94eGdrL2xubHFmc3guaHRt.html
[Status] 页面抓取成功


### Step 2. 解析根页面，提取关于特定年份数据的网址

In [10]:
years = [2021, 2022, 2023, 2024]
keywords = [f"{year}年" for year in years]
soup = BeautifulSoup(response.text, "html.parser")
url_list = []
for a_tag in soup.find_all("a", href=True):
	text = a_tag.get_text(strip=True)
	if any(keyword in text for keyword in keywords):
		new_url = urljoin(base_url, a_tag["href"])
		url_list.append(new_url)
print("\n".join(url_list))

https://join-tsinghua.edu.cn/info/1032/1945.htm
https://join-tsinghua.edu.cn/info/1032/1808.htm
https://join-tsinghua.edu.cn/info/1032/1683.htm
https://join-tsinghua.edu.cn/info/1032/1506.htm


### Step 3. 再次调用wc批量下载静态网页

In [11]:
wc.addmany(url_list)
wc.run()

[Status] 当前工作队列长度：4
[Status] 开始抓取 https://join-tsinghua.edu.cn/info/1032/1945.htm
[Status] 已保存到 thu/aHR0cHM6Ly9qb2luLXRzaW5naHVhLmVkdS5jbi9pbmZvLzEwMzIvMTk0NS5odG0.html
[Status] 页面抓取成功
[Status] Sleeping for 4.07 seconds...
[Status] 当前工作队列长度：3
[Status] 开始抓取 https://join-tsinghua.edu.cn/info/1032/1808.htm
[Status] 已保存到 thu/aHR0cHM6Ly9qb2luLXRzaW5naHVhLmVkdS5jbi9pbmZvLzEwMzIvMTgwOC5odG0.html
[Status] 页面抓取成功
[Status] Sleeping for 3.17 seconds...
[Status] 当前工作队列长度：2
[Status] 开始抓取 https://join-tsinghua.edu.cn/info/1032/1683.htm
[Status] 已保存到 thu/aHR0cHM6Ly9qb2luLXRzaW5naHVhLmVkdS5jbi9pbmZvLzEwMzIvMTY4My5odG0.html
[Status] 页面抓取成功
[Status] Sleeping for 3.78 seconds...
[Status] 当前工作队列长度：1
[Status] 开始抓取 https://join-tsinghua.edu.cn/info/1032/1506.htm
[Status] 已保存到 thu/aHR0cHM6Ly9qb2luLXRzaW5naHVhLmVkdS5jbi9pbmZvLzEwMzIvMTUwNi5odG0.html
[Status] 页面抓取成功
[Status] WebCrawler stopped after 5 pages visited.


## 课堂练习-3-2-3
利用WebCrawler抓取复旦大学近5年录取分数数据，并保存到名为fdu的文件夹中。
+ 目录页 https://ao.fudan.edu.cn/36333/list.htm

## 进阶内容：动态网页抓取

In [ ]:
def get_one_page(bro):
    records = []
    items = bro.eles(".unit-item-padding")
    for item in items:
        try:
            item_link = item.ele("tag:a")
            title = item_link.attr("title")
			href = item_link.attr("href")
			tags = item.ele(".tjb-unit-item-3__info__tags").text
			desp = item.ele(".tjb-unit-item-3__info__desc").text
			ori_price = item.ele(".origion-price").text
			final_price = item.ele(".final-price").text
			records.append((title, href, tags, desp, ori_price,final_price))
		except Exception as e:
			print(f"[Error] {e}")
	return records



In [ ]:
co = ChromiumOptions()
bro = ChromiumPage(co) #启动浏览器
bro.set.auto_handle_alert() #自动确认弹窗
bro.get("https://m.tujia.com/hotel_city73/7-2,101/") #在浏览器中打开目标网页
input("按回车键继续...") #待登录完成后，按回车键继续

item_list = [] #房源列表
max_pages = 10 #待抓取页数上限
start_time = time.time()

for page_idx in range(max_pages):
    res = get_one_page(bro)
    if len(res) == 0: #没有抓到新数据，提前结束
    	break
    item_list += res
    time.sleep(5) #每抓取一个页面后暂停5秒，防止触发反爬机制被封禁
    if time.time()-start_time > 300: #每满5分钟，程序额外休眠1分钟
    	time.sleep(60)
    	start_time = time.time() #重置起始时间
    bro.scroll.to_bottom() #模拟页面滚动到底部，实现自动翻页

#使用Pandas保存为CSV
df = pd.DataFrame(item_list)
df.drop_duplicates(keep='first', inplace=True) #去重：对于重复数据，只保留该数据的首次记录
df.to_csv("minsu-overview.csv", index=False, encoding="utf-8-sig") #指定utf-8-sig编码避免乱码

